# Gene-DINO : DINOv3 Adapted for Transcriptomics gene expression data (for now) as a step 1. 

**Step 1 DINOmics Prototype — Implementation Notebook**

This notebook walks through the complete modified DINOv3 framework for gene expression data.
It covers:
- What was built and what each file does
- Running the pipeline on dummy data (works immediately, no dataset needed)
- Switching to the real HESCAPE dataset (`Peng-AI/hescape-pyarrow`)
- Switching between all four augmentation transforms (choosing which ones)
- Switching between random weights and pretrained Nicheformer weights
- Understanding the training output
- Next steps for scaling to distributed training and adding more transforms from the litterature

> **Note on dataset:** The code in `image_gexp_dataset.py` of hescape hardcodes
> `dataset_path = "Peng-AI/hescape-pyarrow"` and `dataset_name = "human-lung-healthy-panel"`.
> This notebook uses `dataset_path = "Peng-AI/hescape-pyarrow"` and `dataset_name = "human-lung-healthy-panel"` as the real-data example as in the dataloader of Hescape's `image_gexp_dataset.py`.



---
## 1. What Was Built

Four files implement the complete pipeline:

| File | Role | Touches original DINOv3? |
|------|------|-------------------------|
| `nicheformer_backbone.py` | Nicheformer transformer as DINOv3-compatible backbone | No — wraps Nicheformer, satisfies DINOv3 interface |
| `gene_collate.py` | Replaces image collate with gene crop collation | No — output matches `collate_data_and_cast` format exactly |
| `gene_dataset_1.py` | All four augmentation transforms + multi-crop dataset | No — mirrors hescape `image_gexp_dataset.py` logic |
| `train_gene.py` | Teacher-student training loop | Uses original `DINOHead` and `DINOLoss` directly |

### What is unchanged from DINOv3
- `DINOHead` — imported directly from `dinov3.layers.dino_head`
- `DINOLoss` — imported directly from `dinov3.loss.dino_clstoken_loss`
- Sinkhorn-Knopp centering — called directly from `DINOLoss`
- Student temperature sharpening — inside `DINOLoss.forward()`, unchanged
- EMA teacher update formula — identical to `SSLMetaArch.update_ema()`

### Why not use `SSLMetaArch` directly (next step - scaling, now we just making sure the pipeline works with the idea and logic we want)
`SSLMetaArch` requires FSDP and a distributed multi-GPU process group to initialise.
It asserts `cfg.compute_precision.sharding_strategy == 'SHARD_GRAD_OP'` and calls
`ac_compile_parallelize()` before any forward pass. On a single GPU or Colab it crashes
before processing a single batch. 
Nicheformer backbone already satisfies the exact interface `SSLMetaArch` expects.
Plugging it in for distributed training is Step 2 after reviewing this first part (covered at the end of this notebook).


---
## 2. Setup


In [2]:
#print where the script is being run from
import os
print(os.getcwd())

/Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content


In [3]:
if not os.path.exists("./dinov3/"):
    !git clone https://github.com/facebookresearch/dinov3.git

Place all 4 files inside the reposiotry in dinov3 (outside not inside dinov3 there is a folder with name dinvo3 inside the outer folder dinov3)

In [4]:
if not os.path.exists("./nicheformer/"):
    !git clone https://github.com/theislab/nicheformer.git

In [ ]:
if not os.path.exists("./hescape/"):
    !git clone https://github.com/Lalemaouloud/hescape.git ## This is a temporary repo with minor modifications and was cloned from the original https://github.com/peng-lab

Cloning into 'hescape'...
remote: Enumerating objects: 411, done.
remote: Counting objects: 100% (411/411), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 411 (delta 187), reused 407 (delta 186), pack-reused 0 (from 0)
Receiving objects: 100% (411/411), 4.91 MiB | 585.00 KiB/s, done.
Resolving deltas: 100% (187/187), done.


In [8]:
!pwd


/Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content


In [62]:
import os, sys

# ── Set PYTHONPATH so dinov3 package is importable ────────────────────────
DINOV3_ROOT = "/Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content/dinov3/"   # dinov3
if DINOV3_ROOT not in sys.path:
    sys.path.insert(0, DINOV3_ROOT)

# ── Verify the four pipeline files are present ───────────────────────────
required = [
    "nicheformer_backbone.py",
    "gene_collate.py",
    "gene_dataset_1.py",
    "train_gene.py",
]
for f in required:
    path = os.path.join(DINOV3_ROOT, f)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {f:35s} {status}")


  nicheformer_backbone.py             OK
  gene_collate.py                     OK
  gene_dataset_1.py                   OK
  train_gene.py                       OK


In [13]:
# Verify DINOv3 components import correctly
from dinov3.layers.dino_head import DINOHead
from dinov3.loss.dino_clstoken_loss import DINOLoss
print("DINOHead  : OK")
print("DINOLoss  : OK")


from nicheformer_backbone import NicheformerBackbone, build_nicheformer_backbone
from gene_collate import gene_collate_fn
from gene_dataset_1 import make_gene_dataset, GeneExpressionDINODataset
print("nicheformer_backbone : OK")
print("gene_collate         : OK")
print("gene_dataset         : OK")


DINOHead  : OK
DINOLoss  : OK
nicheformer_backbone : OK
gene_collate         : OK
gene_dataset         : OK


---
## 3. The Nicheformer Backbone

The backbone wraps the Nicheformer transformer so it satisfies the DINOv3 backbone interface.

**Architecture:**
- Transformer encoder layers: 12
- Hidden dimension: 512
- Attention heads: 16
- Feedforward dimension: 1024
- Maximum sequence length: 1500
- Vocabulary size: 25000 (random) / 20340 (pretrained)

**Input:** `LongTensor [B, seq_len]` — gene token IDs from any of the four transforms

**Output dict (DINOv3 backbone interface):**
```
x_norm_clstoken    : [B, 512]    ← what DINOHead takes as input
x_norm_patchtokens : [B, 1, 512] ← iBOT stub (disabled in prototype)
x_storage_tokens   : [B, 0, 512] ← register tokens (none in Nicheformer)
```


In [14]:
import torch

# ── Build backbone with RANDOM weights ───────────────────────────────────
student, teacher, embed_dim = build_nicheformer_backbone(checkpoint="none")
print(f"embed_dim = {embed_dim}")
print(f"Parameters: {sum(p.numel() for p in student.parameters()):,}")

# Test forward pass
B, SEQ = 3, 1500
fake_tokens = torch.randint(0, 25000, (B, SEQ), dtype=torch.long)
fake_tokens = fake_tokens * (torch.rand(B, SEQ) > 0.8).long()  # sparse

with torch.no_grad():
    out = student(fake_tokens, is_training=True)

print(f"x_norm_clstoken    : {out['x_norm_clstoken'].shape}")
print(f"x_norm_patchtokens : {out['x_norm_patchtokens'].shape}")
print(f"x_storage_tokens   : {out['x_storage_tokens'].shape}")
print(f"CLS L2 norms       : {out['x_norm_clstoken'].norm(dim=-1)}  (should be 1.0)")


  Building backbone — mode: random weights
  n_tokens=25000  dropout=0.1
  Using random weight initialisation.
embed_dim = 512
Parameters: 38,803,968
x_norm_clstoken    : torch.Size([3, 512])
x_norm_patchtokens : torch.Size([3, 1, 512])
x_storage_tokens   : torch.Size([3, 0, 512])
CLS L2 norms       : tensor([1.0000, 1.0000, 1.0000])  (should be 1.0)


In [17]:
# TO-DO lale : make a clean requirements.txt file. 

In [16]:
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 117.5 kB/s  0:00:06eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 85.9 kB/s  0:00:39 eta 0:00:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [huggingface_hub] [huggingface_hub]


In [ ]:
#HF_TOKEN = os.getenv("HF_TOKEN")
#print(f"HF_TOKEN is {'set' if HF_TOKEN else 'NOT set'}")
HF_TOKEN="..."


HF_TOKEN is NOT set


In [25]:
!pip install safetensors

In [26]:
# ── Build backbone with PRETRAINED weights (downloads ~196MB once) ───────
# Weights from: https://huggingface.co/theislab/Nicheformer
# Note: pretrained model uses n_tokens=20340 (different from random 25000)
#       this is handled automatically

student_pt, teacher_pt, embed_dim = build_nicheformer_backbone(
    checkpoint="theislab/Nicheformer"
)
print(f"Parameters (pretrained): {sum(p.numel() for p in student_pt.parameters()):,}")
print("Pretrained backbone ready.")


  Building backbone — mode: pretrained
  n_tokens=20340  dropout=0.0
  Loading pretrained weights: theislab/Nicheformer
  Loaded 147 weight tensors from checkpoint.
  Skipped 14 head layers (not needed for DINO backbone).
  Embedding weight std = 0.071278  (random xavier ~0.04, pretrained ~0.02-0.09)
  Teacher initialised from pretrained student weights.
Parameters (pretrained): 36,418,048
Pretrained backbone ready.


---
## 4. Augmentation Transforms

Four transforms are implemented for now, matching hescape's `image_gexp_dataset.py` exactly.
For tests and one transform the default is `LogNormOnly` , it requires no reference files and works immediately.


### The gene crop strategy
The DINO augmentation for gene data is `crop_genes()`  it randomly subsamples gene positions:
- **Global crop:** 1500 tokens — large gene panel, seen by teacher + student TO-D0 lale : I need to check if change theses numbers, does it even make sence to have 1500 (wich is the whole token size for nicheformer) as a global crop? 
- **Local crop:** 500 tokens — small gene subset, seen by student only

Each call returns a **different** random subset, this is what forces the model to learn that partial and full gene subsets describe the same cell.


In [27]:
# ── Transform 1: LogNormOnly (default, no files needed) ──────────────────
import torch

# Simulate raw float counts: N cells x G genes, sparse
N, G = 30, 2000
raw_counts = torch.zeros(N, G)
for i in range(N):
    idx = torch.randperm(G)[:400]
    raw_counts[i, idx] = torch.randint(1, 500, (400,)).float()

ds_lognorm = make_gene_dataset(
    raw_counts,
    transform_name   = "lognorm",   # change this to switch testing the transform
    n_global_crops   = 2,
    n_local_crops    = 8,
    global_crop_size = 1500,
    local_crop_size  = 500,
    vocab_size       = 25000,
)

sample = ds_lognorm[0]
print(f"global_crops : {sample['global_crops'].shape}  dtype={sample['global_crops'].dtype}")
print(f"local_crops  : {sample['local_crops'].shape}   dtype={sample['local_crops'].dtype}")

# Two crops from same cell differ
s1, s2 = ds_lognorm[0], ds_lognorm[0]
print(f"Two global crops differ: {not (s1['global_crops'][0] == s2['global_crops'][0]).all().item()}")


global_crops : torch.Size([2, 1500])  dtype=torch.int64
local_crops  : torch.Size([8, 500])   dtype=torch.int64
Two global crops differ: True


In [28]:
!pip install anndata

  Using cached array_api_compat-1.14.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached legacy_api_wrap-1.5-py3-none-any.whl.metadata (2.2 kB)
  Using cached natsort-8.4.0-py3-none-any.whl.metadata (21 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached session_info2-0.4.1-py3-none-any.whl.metadata (2.5 kB)
  Using cached donfig-0.8.1.post1-py3-none-any.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 162.3 kB/s  0:00:51m0:00:0100:03
Using cached array_api_compat-1.14.0-py3-none-any.whl (60 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 188.2 kB/s  0:00:16 eta 0:00:02
Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/20.3 MB 105.1 kB/s eta 0:02:34
Resuming download scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl (4.2 MB/20.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 207.5 kB/

In [63]:
# Change using the full path
os.chdir('/Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content/')
print(f"Now in: {os.getcwd()}")

Now in: /Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content


In [64]:
# ── Transform 2 testing: NicheformerTransform ────────────────────────
# Reference files from the nicheformer repo + dataset folder
#
# model.h5ad and xenium_mean_script.npy are in nicheformer/data/model_means/
# nicheformer_reference.h5ad is in hescape/data/<panel-name>/
#
# Available panels (choose one):
#   hescape/data/human-5k-panel/nicheformer_reference.h5ad
#   hescape/data/human-lung-healthy-panel/nicheformer_reference.h5ad
#   hescape/data/human-breast-panel/nicheformer_reference.h5ad
#   hescape/data/human-immuno-oncology-panel/nicheformer_reference.h5ad
#   hescape/data/human-skin-panel/nicheformer_reference.h5ad
#   hescape/data/human-colon-panel/nicheformer_reference.h5ad
#   hescape/data/human-multi-tissue-panel/nicheformer_reference.h5ad
#
# Technology mean options :
#   xenium_mean_script.npy    <- default
#   cosmx_mean_script.npy
#   merfish_mean_script.npy
#   iss_mean_script.npy
#   dissociated_mean_script.npy

NICHEFORMER_REF = "./nicheformer/data/model_means/model.h5ad"
TECH_MEAN       = "./nicheformer/data/model_means/xenium_mean_script.npy"
DATA_REF        = "./hescape/data/human-5k-panel/nicheformer_reference.h5ad"

import os
if os.path.exists(NICHEFORMER_REF) and os.path.exists(DATA_REF):
    ds_nf = make_gene_dataset(
        raw_counts,
        transform_name             = "nicheformer",
        nicheformer_reference_path = NICHEFORMER_REF,
        technology_mean_path       = TECH_MEAN,
        data_gene_reference_path   = DATA_REF,
    )
    s = ds_nf[0]
    print(f"global_crops : {s['global_crops'].shape}  dtype={s['global_crops'].dtype}")
    print("NicheformerTransform OK")
else:
    print("Reference files not found — update NICHEFORMER_REF and DATA_REF paths above")


global_crops : torch.Size([2, 1500])  dtype=torch.int64
NicheformerTransform OK


In [65]:
# ── Transform 3 testing: scFoundationTransform ──────────────────────
SCF_INDEX = "./dinov3/OS_scRNA_gene_index.19264.tsv"   # update this path
DATA_REF  = "./hescape/data/human-5k-panel/nicheformer_reference.h5ad"

if os.path.exists(SCF_INDEX) and os.path.exists(DATA_REF):
    ds_scf = make_gene_dataset(
        raw_counts,
        transform_name               = "scfoundation",
        scfoundation_gene_index_path = SCF_INDEX,
        data_gene_reference_path     = DATA_REF,
    )
    s = ds_scf[0]
    print(f"global_crops : {s['global_crops'].shape}  dtype={s['global_crops'].dtype}")
    print("scFoundationTransform OK")
else:
    print("Reference files not found — update SCF_INDEX path above")


global_crops : torch.Size([2, 1500])  dtype=torch.int64
scFoundationTransform OK


---
## 5. Run Training on Dummy Data

This verifies the full pipeline end-to-end before touching real data.
Expected output: loss starting near `log(4096) ≈ 8.317` and decreasing each step.


In [66]:
%cd dinov3


/Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content/dinov3


In [16]:
# ── Run with random weights (fastest, no downloads) ──────────────────────
!python train_gene.py \
    --steps      10 \
    --batch_size 3 \
    --device     cuda \
    --checkpoint none \
    --transform  lognorm



  Gene-DINO prototype training
  device       : cuda
  batch_size   : 3
  steps        : 10
  embed_dim    : 512
  prototypes K : 4096
  transform(s) : lognorm
  total views  : 1 transform(s) × 10 crops = 10 (2g + 8l)

  Building backbone — mode: random weights
  n_tokens=25000  dropout=0.1
  Using random weight initialisation.
Student backbone parameters : 38,803,968
Student head parameters     : 6,820,096

Starting training for 10 steps...

step   1/10 | loss=8.3376 | grad_norm=0.172 | t_temp=0.0200 | momentum=0.9960
step   2/10 | loss=8.3361 | grad_norm=0.151 | t_temp=0.0240 | momentum=0.9964
step   3/10 | loss=8.3315 | grad_norm=0.068 | t_temp=0.0280 | momentum=0.9969
step   4/10 | loss=8.3296 | grad_norm=0.043 | t_temp=0.0320 | momentum=0.9973
step   5/10 | loss=8.3284 | grad_norm=0.037 | t_temp=0.0360 | momentum=0.9977
step   6/10 | loss=8.3273 | grad_norm=0.033 | t_temp=0.0400 | momentum=0.9982
step   7/10 | loss=8.3262 | grad_norm=0.029 | t_temp=0.0400 | momentum=0.9986
step  

In [17]:
# ── Run with pretrained Nicheformer weights ───────────────────────────────
# Downloads ~196MB on first run, cached afterwards
!python train_gene.py \
    --steps      10 \
    --batch_size 3 \
    --device     cuda \
    --checkpoint theislab/Nicheformer \
    --transform  lognorm



  Gene-DINO prototype training
  device       : cuda
  batch_size   : 3
  steps        : 10
  embed_dim    : 512
  prototypes K : 4096
  transform(s) : lognorm
  total views  : 1 transform(s) × 10 crops = 10 (2g + 8l)

  Building backbone — mode: pretrained
  n_tokens=20340  dropout=0.0
  Loading pretrained weights: theislab/Nicheformer
  Loaded 147 weight tensors from checkpoint.
  Skipped 14 head layers (not needed for DINO backbone).
  Embedding weight std = 0.071278  (random xavier ~0.04, pretrained ~0.02-0.09)
  Teacher initialised from pretrained student weights.
Student backbone parameters : 36,418,048
Student head parameters     : 6,820,096

Starting training for 10 steps...

step   1/10 | loss=8.3379 | grad_norm=0.175 | t_temp=0.0200 | momentum=0.9960
step   2/10 | loss=8.3362 | grad_norm=0.183 | t_temp=0.0240 | momentum=0.9964
step   3/10 | loss=8.3319 | grad_norm=0.086 | t_temp=0.0280 | momentum=0.9969
step   4/10 | loss=8.3300 | grad_norm=0.054 | t_temp=0.0320 | momentum=0

---
## 5.5 Multi-Transform Mode

What DINOmics logic idea (Augmentation as view) is built on: **multi-transform mode**.

In multi-transform mode,
**all four transforms run simultaneously on the same cell**, each producing its own
2 global + 8 local crops. This creates a stronger self-supervised signal:
the model must learn that the same cell looks the same regardless of which
preprocessing strategy was used, which will eventually reduce the biasing Batch effect.

```
Raw counts [G]
  ├─→ LogNormOnly      → crop → 2 global + 8 local
  ├─→ NormalizeCounts  → crop → 2 global + 8 local
  ├─→ NicheformerTransform → crop → 2 global + 8 local
  └─→ scFoundationTransform → crop → 2 global + 8 local
                                       ─────────────────
  TOTAL: 4 transforms × 10 crops = 40 views per cell
         8 global [1500 tokens] + 32 local [500 tokens]
```

The `--transform` argument accepts one (for testing) or more transforms.
The collate function auto-detects crop counts from the data shape — no other
argument changes needed (At least for now I think).


In [18]:
# ── Single transform (original behaviour, unchanged) ─────────────────────
!python train_gene.py \
    --steps 5 --batch_size 3 --device cuda \
    --transform lognorm


  Gene-DINO prototype training
  device       : cuda
  batch_size   : 3
  steps        : 5
  embed_dim    : 512
  prototypes K : 4096
  transform(s) : lognorm
  total views  : 1 transform(s) × 10 crops = 10 (2g + 8l)

  Building backbone — mode: random weights
  n_tokens=25000  dropout=0.1
  Using random weight initialisation.
Student backbone parameters : 38,803,968
Student head parameters     : 6,820,096

Starting training for 5 steps...

step   1/5 | loss=8.3380 | grad_norm=0.204 | t_temp=0.0200 | momentum=0.9960
step   2/5 | loss=8.3352 | grad_norm=0.138 | t_temp=0.0300 | momentum=0.9970
step   3/5 | loss=8.3317 | grad_norm=0.078 | t_temp=0.0400 | momentum=0.9980
step   4/5 | loss=8.3296 | grad_norm=0.047 | t_temp=0.0400 | momentum=0.9989
step   5/5 | loss=8.3283 | grad_norm=0.037 | t_temp=0.0400 | momentum=0.9999

  Training loop completed successfully.
  Final loss : 8.3283



In [19]:
# ── Two transforms: 20 views (4 global + 16 local) ──────────────────────
!python train_gene.py \
    --steps 5 --batch_size 1 --device cuda \
    --transform lognorm normalize


  Gene-DINO prototype training
  device       : cuda
  batch_size   : 1
  steps        : 5
  embed_dim    : 512
  prototypes K : 4096
  transform(s) : ['lognorm', 'normalize']
  total views  : 2 transform(s) × 10 crops = 20 (4g + 16l)

  Multi-transform mode: 2 transforms × (2g + 8l) crops = 4 global + 16 local = 20 total views per cell
  Building backbone — mode: random weights
  n_tokens=25000  dropout=0.1
  Using random weight initialisation.
Student backbone parameters : 38,803,968
Student head parameters     : 6,820,096

Starting training for 5 steps...

step   1/5 | loss=8.3373 | grad_norm=0.194 | t_temp=0.0200 | momentum=0.9960
step   2/5 | loss=8.3351 | grad_norm=0.152 | t_temp=0.0300 | momentum=0.9970
step   3/5 | loss=8.3312 | grad_norm=0.071 | t_temp=0.0400 | momentum=0.9980
step   4/5 | loss=8.3294 | grad_norm=0.044 | t_temp=0.0400 | momentum=0.9989
step   5/5 | loss=8.3282 | grad_norm=0.035 | t_temp=0.0400 | momentum=0.9999

  Training loop completed successfully.
  Final

In [20]:
# ── All four transforms: 40 views (8 global + 32 local) ─────────────────
# Memory note: use batch_size=1 on a single GPU for 40 views
# On multi-GPU with gradient checkpointing, larger batch sizes work fine
!python train_gene.py \
    --steps 5 --batch_size 1 --device cuda \
    --transform lognorm normalize nicheformer scfoundation


  Gene-DINO prototype training
  device       : cuda
  batch_size   : 1
  steps        : 5
  embed_dim    : 512
  prototypes K : 4096
  transform(s) : ['lognorm', 'normalize', 'nicheformer', 'scfoundation']
  total views  : 4 transform(s) × 10 crops = 40 (8g + 32l)

  Multi-transform mode: 4 transforms × (2g + 8l) crops = 8 global + 32 local = 40 total views per cell
  Building backbone — mode: random weights
  n_tokens=25000  dropout=0.1
  Using random weight initialisation.
Student backbone parameters : 38,803,968
Student head parameters     : 6,820,096

Starting training for 5 steps...

step   1/5 | loss=8.3372 | grad_norm=0.179 | t_temp=0.0200 | momentum=0.9960
step   2/5 | loss=8.3353 | grad_norm=0.145 | t_temp=0.0300 | momentum=0.9970
step   3/5 | loss=8.3313 | grad_norm=0.067 | t_temp=0.0400 | momentum=0.9980
step   4/5 | loss=8.3295 | grad_norm=0.045 | t_temp=0.0400 | momentum=0.9989
step   5/5 | loss=8.3281 | grad_norm=0.037 | t_temp=0.0400 | momentum=0.9999

  Training loop 

In [21]:
# ── All four transforms with pretrained weights ──────────────────────────
!python train_gene.py \
    --steps 5 --batch_size 1 --device cuda \
    --checkpoint theislab/Nicheformer \
    --transform lognorm normalize nicheformer scfoundation



  Gene-DINO prototype training
  device       : cuda
  batch_size   : 1
  steps        : 5
  embed_dim    : 512
  prototypes K : 4096
  transform(s) : ['lognorm', 'normalize', 'nicheformer', 'scfoundation']
  total views  : 4 transform(s) × 10 crops = 40 (8g + 32l)

  Multi-transform mode: 4 transforms × (2g + 8l) crops = 8 global + 32 local = 40 total views per cell
  Building backbone — mode: pretrained
  n_tokens=20340  dropout=0.0
  Loading pretrained weights: theislab/Nicheformer
  Loaded 147 weight tensors from checkpoint.
  Skipped 14 head layers (not needed for DINO backbone).
  Embedding weight std = 0.071278  (random xavier ~0.04, pretrained ~0.02-0.09)
  Teacher initialised from pretrained student weights.
Student backbone parameters : 36,418,048
Student head parameters     : 6,820,096

Starting training for 5 steps...

step   1/5 | loss=8.3376 | grad_norm=0.205 | t_temp=0.0200 | momentum=0.9960
step   2/5 | loss=8.3360 | grad_norm=0.164 | t_temp=0.0300 | momentum=0.9970
st

---
## 5.6 Verify the Multi-Transform Pipeline (inspect_pipeline.py)

The `inspect_pipeline.py` script imports and runs our actual pipeline files to verify
every step. No training, no GPU needed (except the backbone section).

It shows real tensor shapes, real token values, and confirms the DINOv3 interface
contract is satisfied at every stage.

In [77]:
# ── Verify raw cell data ─────────────────────────────────────────────────
!python inspect_pipeline.py --section cell



════════════════════════════════════════════════════════════
  Gene-DINO Pipeline Verification
════════════════════════════════════════════════════════════
  Imports and runs our actual files:
    gene_dataset.py  →  GeneExpressionDINODataset, make_gene_dataset
    gene_collate.py  →  gene_collate_fn
    nicheformer_backbone.py  →  NicheformerBackbone (backbone section only)

  Section: cell
  No training. No GPU needed (except backbone section uses CPU).

════════════════════════════════════════════════════════════
  SECTION 1 — raw gene expression data
════════════════════════════════════════════════════════════

  What you are looking at:
    The raw float tensor produced by make_dummy_data().
    This is what GeneExpressionDINODataset receives as input.
    Shape is [N, G] — N cells, G genes per cell.
    
  Full tensor shape : [9, 2000]  (9 cells, 2000 genes)
  cell 0 (first cell)
    shape  : [2000]
    dtype  : torch.float32
    range  : [0.000, 499.000]
    sample : [0.000, 0.

In [76]:
# ── Verify dataset + transforms (imports gene_dataset.py) ────────────────
!python inspect_pipeline.py --section dataset



════════════════════════════════════════════════════════════
  Gene-DINO Pipeline Verification
════════════════════════════════════════════════════════════
  Imports and runs our actual files:
    gene_dataset.py  →  GeneExpressionDINODataset, make_gene_dataset
    gene_collate.py  →  gene_collate_fn
    nicheformer_backbone.py  →  NicheformerBackbone (backbone section only)

  Section: dataset
  No training. No GPU needed (except backbone section uses CPU).

════════════════════════════════════════════════════════════
  SECTION 2 — GeneExpressionDINODataset  (our actual gene_dataset.py)
════════════════════════════════════════════════════════════

  Importing GeneExpressionDINODataset from gene_dataset.py and running it.
  This is the exact class used during training — not a reimplementation.
    

──────────────────────────────────────────────────
  LogNormOnly transform  (default)
──────────────────────────────────────────────────
  Creating dataset with transform_name='lognorm'...

In [75]:
# ── Verify collate reshaping (imports gene_collate.py) ───────────────────
!python inspect_pipeline.py --section collate



════════════════════════════════════════════════════════════
  Gene-DINO Pipeline Verification
════════════════════════════════════════════════════════════
  Imports and runs our actual files:
    gene_dataset.py  →  GeneExpressionDINODataset, make_gene_dataset
    gene_collate.py  →  gene_collate_fn
    nicheformer_backbone.py  →  NicheformerBackbone (backbone section only)

  Section: collate
  No training. No GPU needed (except backbone section uses CPU).

════════════════════════════════════════════════════════════
  SECTION 3 — gene_collate_fn  (our actual gene_collate.py)
════════════════════════════════════════════════════════════

  Importing gene_collate_fn from gene_collate.py and running a real DataLoader.
  Shows exactly what enters the backbone after collation.
    

──────────────────────────────────────────────────
  1 transform(s) — transform='lognorm'
──────────────────────────────────────────────────

  Batch keys: ['collated_global_crops', 'collated_local_crops', 'c

In [74]:
# ── Verify backbone interface contract (imports nicheformer_backbone.py) ─
!python inspect_pipeline.py --section backbone


════════════════════════════════════════════════════════════
  Gene-DINO Pipeline Verification
════════════════════════════════════════════════════════════
  Imports and runs our actual files:
    gene_dataset.py  →  GeneExpressionDINODataset, make_gene_dataset
    gene_collate.py  →  gene_collate_fn
    nicheformer_backbone.py  →  NicheformerBackbone (backbone section only)

  Section: backbone
  No training. No GPU needed (except backbone section uses CPU).

════════════════════════════════════════════════════════════
  SECTION 4 — NicheformerBackbone  (our actual nicheformer_backbone.py)
════════════════════════════════════════════════════════════

  Importing build_nicheformer_backbone from nicheformer_backbone.py.
  Running a real forward pass through the student backbone.
  Shows input → output shapes and verifies the DINOv3 interface contract.
    
  Building backbone with random weights...
  Building backbone — mode: random weights
  n_tokens=25000  dropout=0.1
  Using random 

In [ ]:
# ── Verify multi-transform mode with reference files ─────────────────────
# Without reference files: runs lognorm + normalize only
#!python inspect_pipeline.py --section multi

# With all reference files (update paths to correct locations):
# 
!python inspect_pipeline.py --section multi \
     --nf_ref      /Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content/nicheformer/data/model_means/model.h5ad \
     --nf_tech     /Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content/nicheformer/data/model_means/xenium_mean_script.npy \
     --nf_data_ref /Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content/hescape/data/human-5k-panel/nicheformer_reference.h5ad \
     --scf_index   /Users/lalemaoulouds/Desktop/MUNICH_STEP1/DINOmics/content/dinov3/OS_scRNA_gene_index.19264.tsv


════════════════════════════════════════════════════════════
  Gene-DINO Pipeline Verification
════════════════════════════════════════════════════════════
  Imports and runs our actual files:
    gene_dataset.py  →  GeneExpressionDINODataset, make_gene_dataset
    gene_collate.py  →  gene_collate_fn
    nicheformer_backbone.py  →  NicheformerBackbone (backbone section only)

  Section: multi
  No training. No GPU needed (except backbone section uses CPU).

════════════════════════════════════════════════════════════
  SECTION 5 — Multi-transform mode 
═══════════════════════════���════════════════════════════════

  Running all 4 transforms simultaneously using our actual gene_dataset.py.
  Shows that the same cell produces 40 different views — 8 global + 32 local.
    
  NicheformerTransform  : files found — INCLUDED
  scFoundationTransform : files found — INCLUDED

  Active transforms : ['lognorm', 'normalize', 'nicheformer', 'scfoundation']
  Views per cell    : 4 transforms x 10 

# Gene-DINO Pipeline Verification

## Overview
Imports and runs our actual files:
- `gene_dataset.py` → `GeneExpressionDINODataset`, `make_gene_dataset`
- `gene_collate.py` → `gene_collate_fn`
- `nicheformer_backbone.py` → `NicheformerBackbone` (backbone section only)

**Section:** multi  
**Mode:** No training. No GPU needed (except backbone section uses CPU).

---

## Section 5 — Multi-transform Mode

Running all 4 transforms simultaneously using our actual `gene_dataset.py`.  
Shows that the same cell produces **40 different views** — 8 global + 32 local.

### Transform Status
- **NicheformerTransform** : files found — INCLUDED
- **scFoundationTransform** : files found — INCLUDED

### Configuration
| Setting | Value |
|---------|-------|
| Active transforms | `['lognorm', 'normalize', 'nicheformer', 'scfoundation']` |
| Views per cell | 4 transforms × 10 crops = **40** (8g + 32l) |
| Global crops per transform | 2 |
| Local crops per transform | 8 |

### Output per Sample (`ds[0]`)

global_crops : [8, 1500] # 4 transforms × 2 global crops
local_crops : [32, 500] # 4 transforms × 8 local crops
Total views : 40 per cell


---

## Are views from different transforms actually different?

| Comparison | Matching Positions | Result |
|------------|-------------------|--------|
| lognorm global 1 vs lognorm global 2 (same transform, diff crop) | 963 / 1500 | Different |
| lognorm global 1 vs normalize global 1 (diff transform, diff crop) | 969 / 1500 | Different |

**Conclusion:** Different transforms create meaningfully different views 

---

## After Collate (`batch_size=3`)

collated_global_crops : [24, 1500] # 8 global crops × 3 cells
collated_local_crops : [96, 500] # 32 local crops × 3 cells
n_global_crops : 8
n_local_crops : 32


### Data Flow to GeneDINO

Teacher sees : [24, 1500]
Student sees : [24, 1500] + [96, 500]


These tensors go directly into `GeneDINO.forward()`

---

## Verification Complete
All sections successfully using our actual code.

### Expected verified output from multi section

```
NicheformerTransform  : files found — INCLUDED
scFoundationTransform : files found — INCLUDED

Active transforms : ['lognorm', 'normalize', 'nicheformer', 'scfoundation']
Views per cell    : 4 transforms x 10 crops = 40 (8g + 32l)

Per-sample output from ds[0]:
  global_crops : [8, 1500]  = 4 transforms × 2 global
  local_crops  : [32, 500]  = 4 transforms × 8 local
  Total views  : 40 per cell

After collate (batch_size=3):
  collated_global_crops : [24, 1500]  = 8 global crops × 3 cells
  collated_local_crops  : [96, 500]   = 32 local crops × 3 cells

Teacher sees : [24, 1500]
Student sees : [24, 1500] + [96, 500]
```

### Memory note (For now)
40 views per cell is compute-intensive. On a single free-tier GPU (Colab):
- Use `--batch_size 1` to avoid OOM
- Use `--dino_out_dim 1024` to reduce prototype head memory




---
## 6. Switch to Real HESCAPE Data to test if the pipeline is working

hescape dataset is hosted at `Peng-AI/hescape-pyarrow` on HuggingFace.
and used in `image_gexp_dataset.py` file:
```python
dataset_path = "Peng-AI/hescape-pyarrow"
dataset_name = "human-lung-healthy-panel"
```

To switch from dummy data to real data, only **three things change**:
1. Replace the `raw_counts` tensor with the HuggingFace dataset
2. Change `transform_name` 
3. Pass the correct reference file paths

The backbone, collate, and training loop are **unchanged**.


In [ ]:
# ── Load real HESCAPE data from HuggingFace ───────────────────────────────
from datasets import load_dataset

# if needed:
# from huggingface_hub import login
# login(token="....")

hf_dataset = load_dataset(
    "Peng-AI/hescape-pyarrow",
    name  = "human-lung-healthy-panel",
    split = "train",
)
print(f"Dataset size : {len(hf_dataset):,} cells")
print(f"Columns      : {hf_dataset.column_names}")


In [ ]:

PANEL      = "human-lung-healthy-panel"  
DATA_REF   = f"hescape/data/{PANEL}/nicheformer_reference.h5ad"

real_dataset = make_gene_dataset(
    hf_dataset,
    transform_name             = "nicheformer",
    nicheformer_reference_path = "./nicheformer/data/model_means/model.h5ad",
    technology_mean_path       = "./nicheformer/data/model_means/xenium_mean_script.npy",
    data_gene_reference_path   = DATA_REF,
    gexp_key                   = "gexp",        # column name in HF dataset
    n_global_crops             = 2,
    n_local_crops              = 8,
    global_crop_size           = 1500,
    local_crop_size            = 500,
)

# Verify one sample
sample = real_dataset[0]
print(f"global_crops : {sample['global_crops'].shape}  dtype={sample['global_crops'].dtype}")
print(f"local_crops  : {sample['local_crops'].shape}   dtype={sample['local_crops'].dtype}")


In [ ]:
# ── DataLoader with gene collate ─────────────────────────────────────────
from torch.utils.data import DataLoader
from functools import partial

real_loader = DataLoader(
    real_dataset,
    batch_size  = 32,          
    shuffle     = True,
    num_workers = 4,
    collate_fn  = partial(gene_collate_fn, n_global_crops=2, n_local_crops=8),
    drop_last   = True,
    pin_memory  = True,
)

batch = next(iter(real_loader))
print(f"collated_global_crops : {batch['collated_global_crops'].shape}")
print(f"collated_local_crops  : {batch['collated_local_crops'].shape}")
print(f"global_batch_size     : {batch['global_batch_size']}")


---
## 7. Training on Real small Data


The `--checkpoint` flag controls which weights to start from.

### Configuration for real data training

```bash
python train_gene.py \
    --steps      1000 \
    --batch_size 32 \
    --device     cuda \
    --checkpoint theislab/Nicheformer \
    --transform  nicheformer scfoundation normalize lognorm 
```

To use the real HuggingFace dataset in `train_gene.py`, replace the
synthetic `raw_counts` block in `train_gene.py` with:

```python
from datasets import load_dataset
hf_dataset = load_dataset(
    "Peng-AI/hescape-pyarrow",
    name  = "human-lung-healthy-panel",
    split = "train",
)
dataset = make_gene_dataset(
    hf_dataset,
    transform_name             = args.transform,
    nicheformer_reference_path = "nicheformer/data/model_means/model.h5ad",
    technology_mean_path       = "nicheformer/data/model_means/xenium_mean_script.npy",
    data_gene_reference_path   = "hescape/data/human-lung-healthy-panel/nicheformer_reference.h5ad",
)
```


In [ ]:
# ── Quick real-data training verification (5 steps) ──────────────────────
# This confirms the full pipeline runs on real cells before long training
!python train_gene.py \
    --steps      5 \
    --batch_size 8 \
    --device     cuda \
    --checkpoint theislab/Nicheformer \
    --transform  lognorm normalize nicheformer scfoundation


---
## 8. Understanding the Training Output

```
step  1/10 | loss=8.3371 | grad_norm=0.194 | t_temp=0.0200 | momentum=0.9960
step  2/10 | loss=8.3355 | grad_norm=0.168 | t_temp=0.0240 | momentum=0.9964
```

| Column | What it means | Healthy range |
|--------|--------------|---------------|
| `loss` | DINO cross-entropy. Maximum = `log(K)` where K=4096 prototypes = **8.317** | Decreasing from ~8.33 |
| `grad_norm` | Gradient norm of student parameters | 0.01 – 5.0. If >10 consistently: reduce LR |
| `t_temp` | Teacher temperature. Warms up 0.02 → 0.04 | Follows schedule |
| `momentum` | EMA momentum. Increases 0.996 → 0.9999 | Follows schedule |

### What healthy training looks like
- Loss decreases every step — model is learning
- Grad norm decreases and stabilises — model is converging
- Loss on **random/dummy data** starts at ~8.33 and barely moves — expected, no real signal
- Loss on **real data** with pretrained weights should start lower (~7.5–8.0) and decrease faster

### Why dummy data loss ≈ real data loss at step 1
The DINO head is randomly initialised in both cases. At step 1, the projection head
maps all embeddings to roughly uniform distributions across 4096 prototypes regardless
of backbone quality. After ~100+ steps on real data, the pretrained backbone advantage
becomes visible in faster convergence.


---
## 9. Quick Reference — All Command-Line Options

```bash
python train_gene.py \
    --steps      10              # number of training steps
    --batch_size 3               # cells per batch
    --device     cuda            # 'cuda' or 'cpu'
    --transform  lognorm         # augmentation: lognorm | normalize | nicheformer | scfoundation
    --checkpoint none            # weights: 'none' | 'theislab/Nicheformer' | '/path/to/file.safetensors'
    --dino_out_dim 4096          # number of DINO prototypes K
```



### Switching weights
| Goal | Command |
|------|---------|
| Random init | `--checkpoint none` |
| Pretrained (HuggingFace) | `--checkpoint theislab/Nicheformer` |
| Pretrained (local file If I decide to use scGPT or another transformer as backbone) | `--checkpoint /path/to/model.safetensors` |


---
## 10. Next Steps — Scaling to Distributed Training

The prototype runs on a single GPU. For large-scale training on our full dataset,
the next step is connecting `NicheformerBackbone` to the original
`SSLMetaArch` from DINOv3.

### What is already done
`NicheformerBackbone` already satisfies the exact interface `SSLMetaArch` expects.
When `get_teacher_output()` or `get_student_output()` call `backbone(images, is_training=True)`,
the backbone returns the correct dict with `x_norm_clstoken`, `x_norm_patchtokens`,
and `x_storage_tokens`. No changes to the backbone are needed.

### What needs to be added

**Step 1 — Register the backbone in `build_model_from_cfg`**

**Step 2 — Override the data augmentation in `SSLMetaArch`**

**Step 3 — Replace `collate_data_and_cast` with `gene_collate_fn`** 

**Step 4 — Run with `torchrun`**

**Step 5 — Run with the appopriate data loader and full data and optimise

### Why this was not done in the prototype 
`SSLMetaArch.__init__` asserts `cfg.compute_precision.sharding_strategy == 'SHARD_GRAD_OP'`
and calls `ac_compile_parallelize()` which requires an initialised distributed process group.

---

## Files Summary

```
dinov3/
├── nicheformer_backbone.py   # Nicheformer as DINOv3 backbone (Step 1-A)
├── gene_collate.py           # Gene crop collation (Step 1-B)
├── gene_dataset.py           # All four transforms + dataset (Step 1-B)
└── train_gene.py             # DINO teacher-student training loop (Step 1-C)
└── inspect_pipeline.py       # Verification
```

All original DINOv3 files are **unchanged**.
`DINOHead` and `DINOLoss` are imported directly from the original codebase.
